# Notebook 7 – Encoding Categorical Variables

This notebook covers converting categorical columns in `customer_transactions_raw.csv` into numeric representations suitable for machine learning, along with guidance on when to use each technique and the risks of choosing the wrong one.

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('customer_transactions_raw.csv')
df.shape

(1000, 12)

In [4]:
df[['gender', 'membership_type', 'payment_method', 'city']].head()

,gender,membership_type,payment_method,city
0,female,Gold,Debit Card,Bengaluru
1,Female,Silver,Net Banking,Delhi
2,Male,Gold,Debit Card,Bengaluru
3,Male,Silver,Credit Card,NaN
4,Male,Silver,Credit Card,Delhi


## 1. What is Categorical Encoding?

Categorical encoding is the process of converting non-numeric category labels into numeric values so that machine learning algorithms, which generally expect numeric input, can use them. The choice of encoding technique affects both model performance and interpretability, and the wrong choice can introduce false relationships that don't exist in the real data.

## 2. Nominal Variables

Nominal variables are categories with **no inherent order** — one value is not "greater" or "less" than another.

**Examples in this dataset:** `payment_method` (Credit Card, Debit Card, UPI, COD...), `city` (Chennai, Delhi, Mumbai...). There is no meaningful ranking among these.

**When to use which encoding:** one-hot/dummy encoding or frequency/target encoding — never label/ordinal encoding, since that would impose a false order.

In [5]:
df['payment_method'].value_counts(dropna=False)

payment_method
Credit Card    265
Debit Card     252
UPI            210
Net Banking    122
COD             97
NaN             34
crypto          20
Name: count, dtype: int64

## 3. Ordinal Variables

Ordinal variables have a **natural, meaningful order**, even though the category labels themselves are not numbers.

**Example in this dataset:** `membership_type` has a clear tier order: Bronze < Silver < Gold < Platinum.

**When to use which encoding:** ordinal encoding (mapping each category to an integer that reflects the order) — this preserves the ranking information that one-hot encoding would discard.

In [6]:
df['membership_type'].value_counts(dropna=False)

membership_type
Bronze      317
Silver      313
Gold        189
Platinum    118
NaN          63
Name: count, dtype: int64

## 4. Binary Variables

Binary variables have exactly **two** possible categories (ignoring missing values).

**Example:** a cleaned `gender` column with only `male`/`female` values is binary.

**When to use which encoding:** a single 0/1 column is sufficient — a full one-hot encoding into two columns is redundant, since the second column is always the complement of the first.

In [7]:
gender_clean = df['gender'].str.strip().str.lower().replace({'m': 'male', 'f': 'female'})
gender_clean.value_counts(dropna=False)

gender
male      480
female    430
NaN        90
Name: count, dtype: int64

In [8]:
df['gender_binary'] = gender_clean.map({'female': 0, 'male': 1})
df[['gender', 'gender_binary']].head(10)

,gender,gender_binary
0,female,0.0
1,Female,0.0
2,Male,1.0
3,Male,1.0
4,Male,1.0
5,male,1.0
6,Male,1.0
7,Male,1.0
8,NaN,NaN
9,Male,1.0


## 5. Label Encoding

Label encoding assigns each unique category an arbitrary integer (0, 1, 2, ...). It is compact but **implies a numeric order that may not exist**.

**When to use:** tree-based models (decision trees, random forests, gradient boosting) that can split on arbitrary thresholds and aren't misled by the artificial ordering. **Avoid for:** linear/distance-based models (linear regression, KNN, SVM) applied to nominal data, since those models will treat the encoded integers as having magnitude and order.

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
payment_clean = df['payment_method'].str.strip().str.lower()
payment_clean_filled = payment_clean.fillna('missing')
df['payment_method_label'] = le.fit_transform(payment_clean_filled)
dict(zip(le.classes_, le.transform(le.classes_)))

{'cod': np.int64(0),
 'credit card': np.int64(1),
 'crypto': np.int64(2),
 'debit card': np.int64(3),
 'missing': np.int64(4),
 'net banking': np.int64(5),
 'upi': np.int64(6)}

## 6. Ordinal Encoding

Ordinal encoding is like label encoding, but the integers are assigned **deliberately** to reflect a known real-world order, rather than arbitrarily.

**When to use:** any genuinely ordinal variable, such as `membership_type` here, education level, or satisfaction ratings (low/medium/high). This preserves useful rank information for the model.

In [10]:
membership_order = {'bronze': 1, 'silver': 2, 'gold': 3, 'platinum': 4}
membership_clean = df['membership_type'].str.strip().str.lower()
df['membership_ordinal'] = membership_clean.map(membership_order)
df[['membership_type', 'membership_ordinal']].drop_duplicates()

,membership_type,membership_ordinal
0,Gold,3.0
1,Silver,2.0
6,Platinum,4.0
9,NaN,NaN
14,Bronze,1.0


## 7. One-Hot Encoding

One-hot encoding creates a separate binary (0/1) column for each category. No category is implied to be numerically greater than another, which makes it the safest default for nominal variables with a manageable number of categories.

**When to use:** nominal variables with low-to-moderate cardinality (roughly under 10–15 unique values), especially for linear models, logistic regression, and neural networks.

In [11]:
payment_onehot = pd.get_dummies(payment_clean, prefix='payment')
payment_onehot.head()

,payment_cod,payment_credit card,payment_crypto,payment_debit card,payment_net banking,payment_upi
0,False,False,False,True,False,False
1,False,False,False,False,True,False
2,False,False,False,True,False,False
3,False,True,False,False,False,False
4,False,True,False,False,False,False


## 8. Dummy Variables

Dummy encoding is one-hot encoding with **one category dropped** to avoid the "dummy variable trap" — perfect multicollinearity, where one column can be exactly predicted from the others (since they must sum to 1 across a row). This matters for linear regression and other models that assume no perfect collinearity among predictors.

**When to use:** the same cases as one-hot encoding, but specifically when feeding the result into a linear model that requires independent predictors.

In [12]:
payment_dummies = pd.get_dummies(payment_clean, prefix='payment', drop_first=True)
payment_dummies.head()

,payment_credit card,payment_crypto,payment_debit card,payment_net banking,payment_upi
0,False,False,True,False,False
1,False,False,False,True,False
2,False,False,True,False,False
3,True,False,False,False,False
4,True,False,False,False,False


## 9. Frequency Encoding

Frequency encoding replaces each category with how often it appears in the data (count or proportion). It keeps the feature to a single numeric column, which is useful when cardinality is too high for one-hot encoding, though it can accidentally group unrelated categories that happen to occur equally often.

**When to use:** high-cardinality nominal variables like `city` here, when one-hot encoding would create too many sparse columns and there's no external target-leakage concern.

In [13]:
city_clean = df['city'].str.strip().str.lower()
city_freq = city_clean.value_counts(normalize=True)
df['city_freq_encoded'] = city_clean.map(city_freq)
df[['city', 'city_freq_encoded']].head(10)

,city,city_freq_encoded
0,Bengaluru,0.203070
1,Delhi,0.187721
2,Bengaluru,0.203070
3,NaN,NaN
4,Delhi,0.187721
5,chennai,0.211334
6,Bengaluru,0.203070
7,Chennai,0.211334
8,delhi,0.187721
9,NaN,NaN


## 10. Target Encoding

Target encoding replaces each category with a statistic (commonly the mean) of the target variable for that category. It can capture strong predictive signal, but it directly uses the target, so it **must be computed only on training data with cross-validation/smoothing** to avoid data leakage and overfitting, especially on rare categories.

**When to use:** high-cardinality nominal variables in a supervised learning setting where frequency encoding underperforms and you have enough data per category to estimate a reliable mean, along with careful validation-fold handling.

In [14]:
purchase_amount_clean = df['purchase_amount'].astype(str).str.replace('$', '', regex=False)
df['purchase_amount_clean'] = pd.to_numeric(purchase_amount_clean, errors='coerce')

city_target_mean = df.groupby(city_clean)['purchase_amount_clean'].transform('mean')
df['city_target_encoded'] = city_target_mean
df[['city', 'purchase_amount_clean', 'city_target_encoded']].head(10)

,city,purchase_amount_clean,city_target_encoded
0,Bengaluru,99.13,200.838547
1,Delhi,113.23,124.129874
2,Bengaluru,248.19,200.838547
3,NaN,51.27,NaN
4,Delhi,97.97,124.129874
5,chennai,76.87,260.820112
6,Bengaluru,183.44,200.838547
7,Chennai,110.33,260.820112
8,delhi,41.88,124.129874
9,NaN,162.85,NaN


In [15]:
global_mean = df['purchase_amount_clean'].mean()
category_counts = city_clean.map(city_clean.value_counts())
smoothing_weight = 10
smoothed_target_encoding = (
    (category_counts * df.groupby(city_clean)['purchase_amount_clean'].transform('mean') + smoothing_weight * global_mean)
    / (category_counts + smoothing_weight)
)
df['city_target_encoded_smoothed'] = smoothed_target_encoding
df[['city', 'city_target_encoded', 'city_target_encoded_smoothed']].head(10)

,city,city_target_encoded,city_target_encoded_smoothed
0,Bengaluru,200.838547,202.911916
1,Delhi,124.129874,130.901709
2,Bengaluru,200.838547,202.911916
3,NaN,NaN,NaN
4,Delhi,124.129874,130.901709
5,chennai,260.820112,259.643062
6,Bengaluru,200.838547,202.911916
7,Chennai,260.820112,259.643062
8,delhi,124.129874,130.901709
9,NaN,NaN,NaN


## 11. Handling Unknown Categories

A category that appears in new/production data but was never seen during training will break encoders that only know the training-time categories (label encoders raise errors; one-hot encoders silently produce all-zero rows).

**Strategies:**
- Reserve an explicit `"unknown"` / `"other"` bucket during training for rare or held-out categories.
- Use `handle_unknown='ignore'` in scikit-learn's `OneHotEncoder` to emit an all-zero row instead of raising an error.
- For frequency/target encoding, fall back to the overall mean or zero frequency for unseen categories.

In [17]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(payment_clean_filled.to_frame())
new_data = pd.DataFrame({'payment_method': ['debit card', 'cheque']})
ohe.transform(new_data)

array([[0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.]])

## 12. High-Cardinality Categories

High-cardinality categorical variables (many unique values, such as customer ID, ZIP code, or product SKU) cause problems for one-hot encoding: the resulting matrix becomes very wide and sparse, increasing memory use and the risk of overfitting.

**Strategies:** frequency encoding, target encoding (with proper regularization), grouping rare categories into an "other" bucket, or hashing encoding for very large category spaces.

In [18]:
city_clean.nunique(), city_clean.value_counts().tail(10)

(5,
 city
 chennai      179
 mumbai       179
 bengaluru    172
 delhi        159
 hyderabad    158
 Name: count, dtype: int64)

In [19]:
min_count_threshold = 10
category_freq_counts = city_clean.value_counts()
rare_categories = category_freq_counts[category_freq_counts < min_count_threshold].index
city_grouped = city_clean.where(~city_clean.isin(rare_categories), other='other')
city_grouped.value_counts()

city
chennai      179
mumbai       179
bengaluru    172
delhi        159
hyderabad    158
Name: count, dtype: int64

## 13. Risks of Inappropriate Encoding

- **Imposing false order (label/ordinal encoding on nominal data):** encoding `payment_method` as Credit Card=0, Debit Card=1, UPI=2 tells a linear model that UPI is "twice" Debit Card, which is meaningless and can distort coefficients and distance calculations.
- **Dummy variable trap:** using full one-hot encoding (without dropping a column) in a linear regression introduces perfect multicollinearity, making coefficients unstable or the model unsolvable.
- **Dimensionality explosion:** one-hot encoding a high-cardinality column like `city` (or worse, `customer_id`) can create hundreds or thousands of mostly-empty columns, slowing training and encouraging overfitting.
- **Target leakage:** naive target encoding computed on the full dataset (including validation/test rows) leaks target information into the features, producing artificially strong validation performance that collapses in production.
- **Unseen categories at inference time:** an encoder trained only on known categories can crash or silently misencode a new category seen in production, if unknown-category handling isn't built in.
- **Losing information through over-simplification:** frequency encoding two unrelated categories that happen to have the same count will assign them the identical encoded value, hiding real distinctions between them.
- **Ignoring true category order:** one-hot encoding a genuinely ordinal variable like `membership_type` throws away the rank relationship (Platinum being 'above' Gold), which may weaken a model that could have benefited from that ordering.

In [17]:
encoding_summary = pd.DataFrame({
    'variable': ['gender', 'membership_type', 'payment_method', 'city'],
    'variable_type': ['binary', 'ordinal', 'nominal (low cardinality)', 'nominal (high cardinality)'],
    'recommended_encoding': ['binary 0/1', 'ordinal encoding', 'one-hot / dummy encoding', 'frequency or target encoding'],
})
encoding_summary

,variable,variable_type,recommended_encoding
0,gender,binary,binary 0/1
1,membership_type,ordinal,ordinal encoding
2,payment_method,nominal (low cardinality),one-hot / dummy encoding
3,city,nominal (high cardinality),frequency or target encoding


In [18]:
df.to_csv('customer_transactions_encoded.csv', index=False)
df.shape

(1000, 19)